In [14]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator

In [15]:
load_dotenv()

True

In [16]:
genarator_llm = ChatGroq(
    model='openai/gpt-oss-20b'
)

evaluator_llm = ChatGroq(
    model = 'qwen/qwen3.6-27b'
)

optimizer_llm = ChatGroq(
    model = 'groq/compound'
)

In [17]:
#state
class TweetState(TypedDict):

    topic: str
    tweet: str
    evaluation : Literal["approved", "needs_improvement"]
    feedback : str
    iteration : int
    max_iteration : int

In [ ]:
def gen_tweet(state: TweetState):

    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
""")
    ]

    response = genarator_llm.invoke(messages).content

    return {'tweet' : response}


In [ ]:
def eval_tweet(state: TweetState):
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
    ]

In [20]:
graph = StateGraph(TweetState)

graph.add_node('gen', gen_tweet)
graph.add_node('eval', eval_tweet)
graph.add_node('optm', optm_tweet)

NameError: name 'eval_tweet' is not defined